# 🚗 Smart Garage — Навчання персональної моделі (Qwen 2.5 Coder 1.5B LoRA)
Цей блокнот донавчає компактну модель під українську мову та виклики команд розумного гаража (без вигадування неіснуючих датчиків воріт чи газу).

**Вимоги:** Безкоштовний GPU T4 (Меню: `Середовище виконання` -> `Змінити тип середовища виконання` -> `T4 GPU`).
**Час навчання:** ~10-12 хвилин.

In [ ]:
# 1. Встановлення Unsloth та бібліотек
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 2. Завантаження датасету (Завантажте файл train.jsonl у панель файлів Colab зліва)
import os
if not os.path.exists('train.jsonl'):
    print('⚠️ Будь ласка, перетягніть файл train.jsonl у панель файлів Colab!')
else:
    print('✅ Файл train.jsonl знайдено!')

In [ ]:
# 3. Завантаження базової моделі Qwen 2.5 Coder 1.5B Instruct
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Додавання адаптерів LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print('✅ Модель готова до тренування!')

In [ ]:
# 4. Підготовка даних
import json
from datasets import Dataset

records = []
with open("train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

formatted_texts = []
for r in records:
    text = tokenizer.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=False)
    formatted_texts.append(text)

dataset = Dataset.from_dict({"text": formatted_texts})
print(f'✅ Підготовлено {len(dataset)} прикладів.')

In [ ]:
# 5. Запуск тренування
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs",
    ),
)

print('🚀 Старт тренування (~10 хв)...')
trainer.train()
print('🎉 Тренування завершено!')

In [ ]:
# 6. Експорт у квантований GGUF (Q4_K_M)
print('📦 Конвертація у GGUF...')
model.save_pretrained_gguf(
    "smartgarage-qwen1.5b-uk",
    tokenizer,
    quantization_method="q4_k_m"
)
print('✅ Готово! Знайдіть у списку файлів smartgarage-qwen1.5b-uk-q4_k_m.gguf та завантажте його.')